In [1]:
import json
import pandas as pd

In [2]:
filename = "ecko_k10_IC"
# filename = "ecko_k10_shortest"
# filename = "primekg_k10_IC"
# filename = "primekg_k10_shortest"

filepath = "./path-files/" + filename + ".json"

In [3]:
# props
has_mutation = "http://purl.obolibrary.org/obo/VO_0001039" # has gene mutation
has_mutation_inverse = "https://katy-project.eu/gene_mutation_of" # gene mutation of
inhibits = "https://raw.githubusercontent.com/liseda-lab/ImPO/main/ImPO.owl#inhibits" # inhibits
equivalent_class = "http://www.w3.org/2002/07/owl#equivalentClass" # equivalent class

In [4]:
def path_has_mapping(path):
    if "equivalent" in str(path):
        return True
    elif ("inhibits" in str(path) 
        or "inhibited_by" in str(path) 
        or "inhibited by" in str(path)):
            return True
    elif ("has gene mutation" in str(path) 
        or "has_gene_mutation" in str(path) 
        or "VO_0001039" in str(path) 
        or "gene mutation of" in str(path) 
        or "gene_mutation_of" in str(path)):
            return True
    else:
        return False

In [5]:
def mapping_types(path):
    text = " ".join(str(p).lower() for p in path)
    types = []
    if "equivalent" in text:
        types.append("e")
    if any(x in text for x in ["inhibits", "inhibited_by", "inhibited by"]):
        types.append("i")
    if "gene_mutation_of" in text or "vo_0001039" in text:
        types.append("m")
    return types

In [6]:
def get_values_from_json(value):
    path = value['path']
    gene = path[0].replace(" gene", "")
    drug = path[-1]
    if "avg_weight" in value:
        weight = value['avg_weight']
    else:
        weight = 0
    return gene, drug, weight, path

In [7]:
def stats(paths, weights):
    path_lengths = []
    for path in paths:
        path_lengths.append(len(path))
    avg_length = sum(path_lengths)/len(path_lengths)
    avg_weight = sum(weights)/len(weights)
    return avg_length, avg_weight, max(path_lengths), max(weights), min(path_lengths), min(weights)

In [8]:
def stats(df, mappings):
    # unique - gene, drug, weights, paths, pairs
    print("genes: ", df['gene'].nunique())
    print("drugs: ", df['drug'].nunique())
    print("paths: ", df['path'].apply(tuple).nunique())
    print("rows: ", len(df.index))
    print("unique pairs: ", df[['gene', 'drug']].drop_duplicates().shape[0])
    
    # calcs weight
    print("---")
    print("max weight: ", df["weight"].max())
    print("avg weight: ", df.loc[:, 'weight'].mean())
    print("min weight: ", df["weight"].min())

    # calcs length
    lengths = df['path'].apply(len)
    print("---")
    print("max length: ", lengths.max())
    print("avg length: ", lengths.mean())
    print("min length: ", lengths.min())
    
    # inhibs, equivs, muts
    if mappings:
        mapping_counts = (
            df["path"]
            .apply(mapping_types)
            .explode()
            .value_counts()
        )

        print("---")
        print("equivs:", mapping_counts.get("e", 0))
        print("inhibs:", mapping_counts.get("i", 0))
        print("muts:", mapping_counts.get("m", 0))

In [9]:
def find_best_paths(df):
    best_df = df.loc[df.groupby(['gene', 'drug'])['weight'].idxmax()].reset_index(drop=True)
    return best_df

In [10]:
def separate_df_by_mappings(df):
    with_list = []
    wo_list = []
    for index, row in df.iterrows():
        if path_has_mapping(row['path']):
            with_list.append(row)
        else:
            wo_list.append(row)
    df_with = pd.DataFrame(with_list).reset_index(drop=True)
    df_wo = pd.DataFrame(wo_list).reset_index(drop=True)
    return df_with, df_wo

In [11]:
def parse(json_file):
    rows = []
    with open(json_file) as f:
        dict_json = json.load(f)
        for item in dict_json:
            for value in dict_json[item]:
                gene, drug, weight, path = get_values_from_json(value)
                rows.append({
                    "gene": gene,
                    "drug": drug,
                    "weight": weight,
                    "path": path
                })  
    return pd.DataFrame(rows)

In [12]:
df = parse(filepath)
df

,gene,drug,weight,path
0,http://purl.obolibrary.org/obo/NCIT_C20872,http://purl.obolibrary.org/obo/NCIT_C68814,0.498324,"[http://purl.obolibrary.org/obo/NCIT_C20872, h..."
1,http://purl.obolibrary.org/obo/NCIT_C20872,http://purl.obolibrary.org/obo/NCIT_C68814,0.646248,"[http://purl.obolibrary.org/obo/NCIT_C20872, h..."
2,http://purl.obolibrary.org/obo/NCIT_C20872,http://purl.obolibrary.org/obo/NCIT_C68814,0.429740,"[http://purl.obolibrary.org/obo/NCIT_C20872, h..."
3,http://purl.obolibrary.org/obo/NCIT_C20872,http://purl.obolibrary.org/obo/NCIT_C68814,0.608860,"[http://purl.obolibrary.org/obo/NCIT_C20872, h..."
4,http://purl.obolibrary.org/obo/NCIT_C20872,http://purl.obolibrary.org/obo/NCIT_C68814,0.529150,"[http://purl.obolibrary.org/obo/NCIT_C20872, h..."
...,...,...,...,...
2645,http://purl.obolibrary.org/obo/NCIT_C178165,http://purl.obolibrary.org/obo/NCIT_C38718,0.550329,"[http://purl.obolibrary.org/obo/NCIT_C178165, ..."
2646,http://purl.obolibrary.org/obo/NCIT_C178165,http://purl.obolibrary.org/obo/NCIT_C38718,0.546734,"[http://purl.obolibrary.org/obo/NCIT_C178165, ..."
2647,http://purl.obolibrary.org/obo/NCIT_C178165,http://purl.obolibrary.org/obo/NCIT_C38718,0.546734,"[http://purl.obolibrary.org/obo/NCIT_C178165, ..."
2648,http://purl.obolibrary.org/obo/NCIT_C178165,http://purl.obolibrary.org/obo/NCIT_C38718,0.538340,"[http://purl.obolibrary.org/obo/NCIT_C178165, ..."


In [13]:
df_with, df_wo = separate_df_by_mappings(df)

In [14]:
best_df = find_best_paths(df)
best_df

,gene,drug,weight,path
0,http://purl.obolibrary.org/obo/NCIT_C101363,http://purl.obolibrary.org/obo/NCIT_C116870,0.600190,"[http://purl.obolibrary.org/obo/NCIT_C101363, ..."
1,http://purl.obolibrary.org/obo/NCIT_C101363,http://purl.obolibrary.org/obo/NCIT_C38718,0.546845,"[http://purl.obolibrary.org/obo/NCIT_C101363, ..."
2,http://purl.obolibrary.org/obo/NCIT_C101718,http://purl.obolibrary.org/obo/NCIT_C116870,0.613265,"[http://purl.obolibrary.org/obo/NCIT_C101718, ..."
3,http://purl.obolibrary.org/obo/NCIT_C101718,http://purl.obolibrary.org/obo/NCIT_C38718,0.603227,"[http://purl.obolibrary.org/obo/NCIT_C101718, ..."
4,http://purl.obolibrary.org/obo/NCIT_C102464,http://purl.obolibrary.org/obo/NCIT_C116870,0.619064,"[http://purl.obolibrary.org/obo/NCIT_C102464, ..."
...,...,...,...,...
260,http://purl.obolibrary.org/obo/NCIT_C97244,http://purl.obolibrary.org/obo/NCIT_C38718,0.562866,"[http://purl.obolibrary.org/obo/NCIT_C97244, h..."
261,http://purl.obolibrary.org/obo/NCIT_C97433,http://purl.obolibrary.org/obo/NCIT_C116870,0.620600,"[http://purl.obolibrary.org/obo/NCIT_C97433, h..."
262,http://purl.obolibrary.org/obo/NCIT_C97433,http://purl.obolibrary.org/obo/NCIT_C38718,0.651466,"[http://purl.obolibrary.org/obo/NCIT_C97433, h..."
263,http://purl.obolibrary.org/obo/NCIT_C99608,http://purl.obolibrary.org/obo/NCIT_C116870,0.612618,"[http://purl.obolibrary.org/obo/NCIT_C99608, h..."


In [15]:
best_df_with, best_df_wo = separate_df_by_mappings(best_df)

In [16]:
print("\n--- ALL ---")
stats(df, True)
df.to_csv("./dataframes/" + filename + '_all.tsv', sep = '\t', encoding='utf-8', index=False)
print("\n--- with mappings ---")
stats(df_with, True)
df_with.to_csv("./dataframes/" + filename + '_all-with.tsv', sep = '\t', encoding='utf-8', index=False)
print("\n--- without mappings ---")
stats(df_wo, False)
df_wo.to_csv("./dataframes/" + filename + '_all-wo.tsv', sep = '\t', encoding='utf-8', index=False)

print("\n--- BEST ---")
stats(best_df, True)
best_df.to_csv("./dataframes/" + filename + '_best.tsv', sep = '\t', encoding='utf-8', index=False)
print("\n--- with mappings ---")
stats(best_df_with, True)
best_df_with.to_csv("./dataframes/" + filename + '_best-with.tsv', sep = '\t', encoding='utf-8', index=False)
print("\n--- without mappings ---")
stats(best_df_wo, False)
best_df_wo.to_csv("./dataframes/" + filename + '_best-wo.tsv', sep = '\t', encoding='utf-8', index=False)


--- ALL ---
genes:  131
drugs:  5
paths:  2650
rows:  2650
unique pairs:  265
---
max weight:  0.7539678728995288
avg weight:  0.5634238738877289
min weight:  0.42820226885761226
---
max length:  19
avg length:  10.855849056603773
min length:  9
---
equivs: 202
inhibs: 1314
muts: 21

--- with mappings ---
genes:  131
drugs:  5
paths:  1422
rows:  1422
unique pairs:  176
---
max weight:  0.7539678728995288
avg weight:  0.5429413431986923
min weight:  0.4333188697974142
---
max length:  19
avg length:  9.766526019690577
min length:  9
---
equivs: 202
inhibs: 1314
muts: 21

--- without mappings ---
genes:  127
drugs:  4
paths:  1228
rows:  1228
unique pairs:  140
---
max weight:  0.6742494007829656
avg weight:  0.5871422441155874
min weight:  0.42820226885761226
---
max length:  17
avg length:  12.117263843648209
min length:  9

--- BEST ---
genes:  131
drugs:  5
paths:  265
rows:  265
unique pairs:  265
---
max weight:  0.7539678728995288
avg weight:  0.6036951210372938
min weight:  0.4